# tsim timing check for magic state cultivation circuit

Generate the `end2end-inplace-distillation` circuit with S/S_DAG replaced by T/T_DAG,
then use tsim to simulate and measure sampling time.

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

In [2]:
import pathlib
import re
import sys
import time

# --- project src ---
src_path = pathlib.Path("__file__").parent.parent / "src"
src_path = pathlib.Path().resolve().parent / "src"
assert src_path.exists(), f"src path not found: {src_path}"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# --- tsim (cloned repo) ---
tsim_src = pathlib.Path().resolve().parent.parent / "tsim" / "src"
assert tsim_src.exists(), f"tsim src not found: {tsim_src}"
if str(tsim_src) not in sys.path:
    sys.path.insert(0, str(tsim_src))

import cultiv
import gen
import tsim

print(f"tsim version : {tsim.__version__}")
print(f"src path     : {src_path}")
print(f"tsim src path: {tsim_src}")

tsim version : 0.1.0
src path     : /home/bobxu_0712/Documents/quantum_research/magic-state-cultivation/src
tsim src path: /home/bobxu_0712/Documents/quantum_research/tsim/src


## 1. Circuit parameters

In [10]:
# Circuit parameters — same as normal usage, only d2 changed to 9
D1            = 3          # color code distance (dcolor)
D2            = 11          # surface code distance (dsurface)
BASIS         = 'Y'
INJECT_STYLE  = 'unitary'
R_GROWING     = D1         # r_growing = r_in_escape
R_END         = 0
NOISE_STR     = 1e-3

print(f"d1={D1}, d2={D2}, basis={BASIS}, inject={INJECT_STYLE}, r_growing={R_GROWING}, p={NOISE_STR}")

d1=3, d2=11, basis=Y, inject=unitary, r_growing=3, p=0.001


## 2. Build ideal circuit and add SI1000 noise

In [11]:
ideal_circuit = cultiv.make_end2end_cultivation_circuit(
    dcolor=D1,
    dsurface=D2,
    basis=BASIS,
    r_growing=R_GROWING,
    r_end=R_END,
    inject_style=INJECT_STYLE,
)

# Transpile to CZ (Z-basis interactions) then apply SI1000 noise
import gen
cz_circuit  = gen.transpile_to_z_basis_interaction_circuit(ideal_circuit)
noise_model = gen.NoiseModel.si1000(NOISE_STR)
noisy_circuit = noise_model.noisy_circuit_skipping_mpp_boundaries(cz_circuit)

print(f"Qubits     : {noisy_circuit.num_qubits}")
print(f"Detectors  : {noisy_circuit.num_detectors}")
print(f"Instructions: {len(list(noisy_circuit.flattened()))}")

Qubits     : 246
Detectors  : 561
Instructions: 1302


## 3. Replace S / S_DAG → T / T_DAG

In [12]:
def replace_s_with_t(circuit_str: str) -> str:
    """Replace S_DAG -> T_DAG and S -> T (word-boundary safe)."""
    # Replace S_DAG first to avoid partial match with S
    result = re.sub(r'\bS_DAG\b', 'T_DAG', circuit_str)
    result = re.sub(r'\bS\b', 'T', result)
    return result

original_str  = str(noisy_circuit)
modified_str  = replace_s_with_t(original_str)

# Sanity-check: count substitutions
n_s     = len(re.findall(r'\bS\b',     original_str))
n_sdag  = len(re.findall(r'\bS_DAG\b', original_str))
n_t     = len(re.findall(r'\bT\b',     modified_str))
n_tdag  = len(re.findall(r'\bT_DAG\b', modified_str))
print(f"Original : S={n_s}, S_DAG={n_sdag}")
print(f"Modified : T={n_t}, T_DAG={n_tdag}")

Original : S=0, S_DAG=0
Modified : T=0, T_DAG=0


## 4. Build tsim circuit

In [14]:
t0 = time.perf_counter()
tsim_circuit = tsim.Circuit(modified_str)
t1 = time.perf_counter()

print(f"tsim.Circuit() parse time: {t1 - t0:.3f} s")
print(f"Qubits    : {tsim_circuit.num_qubits}")
print(f"Detectors : {tsim_circuit.num_detectors}")

tsim.Circuit() parse time: 0.012 s
Qubits    : 246
Detectors : 561


## 5. Compile detector sampler (includes JIT compilation on first call)

In [15]:
t0 = time.perf_counter()
sampler = tsim_circuit.compile_detector_sampler()
t1 = time.perf_counter()
print(f"compile_detector_sampler() time: {t1 - t0:.3f} s")

compile_detector_sampler() time: 56.379 s


## 6. Sample — warm-up shot (triggers JIT compilation)

In [16]:
WARMUP_SHOTS = 10

t0 = time.perf_counter()
_ = sampler.sample(
    shots=WARMUP_SHOTS,
    use_detector_reference_sample=True,
    use_observable_reference_sample=True,
)
t1 = time.perf_counter()
print(f"Warm-up ({WARMUP_SHOTS} shots, includes JIT): {t1 - t0:.3f} s")

Warm-up (10 shots, includes JIT): 2.652 s


## 7. Sample — timed benchmark

In [17]:
BENCHMARK_SHOTS = 1000

t0 = time.perf_counter()
samples = sampler.sample(
    shots=BENCHMARK_SHOTS,
    use_detector_reference_sample=True,
    use_observable_reference_sample=True,
)
t1 = time.perf_counter()

elapsed = t1 - t0
throughput = BENCHMARK_SHOTS / elapsed
print(f"Benchmark : {BENCHMARK_SHOTS} shots in {elapsed:.3f} s  ({throughput:.1f} shots/s)")
print(f"Sample shape: {samples.shape}")

Benchmark : 1000 shots in 1.863 s  (536.6 shots/s)
Sample shape: (1000, 561)
